### Objetivos:
- Realizar as etapas de pré-processamento
  > Dupla 1 (Gabriel Ferreira & Gabriel Nogueira)
  * Balanceamento (2 técnicas de balanceamento)
  * Tratamento de dados ausentes (1 técnica de conversão simbólico-numérica, devidamente justificada)
  * Dados redundantes
  * Dados inconsistentes
  * Conversão simbólica-numérica
  
  > Dupla 2 (Daniel Santos & Rafael)
  * Transformação de atributos numéricos
  * Redução de dimensionalidade
  * Seleção de instâncias
  * Eliminação de outliers

- Comparar o desempenho de diferentes tipos de classificação

# Etapa 01
## Tarefas:
> Dupla 01 (Gabriel F. & Gabriel N.)
- Caracterização da base NSL-KDD;
- Identificação das classes
- Identificação de desbalanceamento

> Dupla 02 (Daniel S. & Daniel R. & Rafael)
- Identificação de atributos categóricos
- Identificação de inconsistências
- Identificação de redundâncias
- Identificação de possíveis problemas de vazamento de dados.

## Importação de bibliotecas e algoritmos de classificação

### Importação do SMOTE

In [1]:
# !pip install -U imbalanced-learn # GN: Import

In [2]:
import pandas as pd                       # GN: Biblioteca padrão para leitura e tratamento de dados
from sklearn.impute import KNNImputer     # GN: Biblioteca padrão para imputação de dados
from imblearn.over_sampling import SMOTE  # GN: Biblioteca padrão para balanceamento de dados

## Caracterização da Base


In [3]:
# GN: Definição dos atributos da base
#     A o atributo de classificação "class" foi trocado para "target" a fins de normalização
# GN: Carrega a base e não lê os dados cabeçalho

caminho_arquivo = "/content/sample_data/treino.csv"
base = pd.read_csv(caminho_arquivo)
base = base.rename(columns={"class": "target"})


if "target" not in base.columns or not base["target"].isin(["normal", "anomaly"]).all():
    raise ValueError("O treino deve conter class com valores normal ou anomaly.")

print("CARACTERIZAÇÃO DA BASE DE DADOS")

# Quantidade de instâncias e atributos da base
num_instancias, num_atributos = base.shape
print("Quantidade de instâncias: ", num_instancias)
print("Número total de colunas: ", num_atributos)
print("Atributos descritivos: ", num_atributos - 1)

print("\nTIPOS DE DADOS PRESENTES")
# Mapeamento do tipo de dado por coluna
print(base.dtypes)

CARACTERIZAÇÃO DA BASE DE DADOS
Quantidade de instâncias:  69833
Número total de colunas:  50
Atributos descritivos:  49

TIPOS DE DADOS PRESENTES
duration                         int64
protocol_type                   object
service                         object
flag                            object
src_bytes                      float64
dst_bytes                        int64
land                             int64
wrong_fragment                   int64
urgent                           int64
hot                              int64
num_failed_logins                int64
logged_in                      float64
num_compromised                float64
root_shell                     float64
su_attempted                   float64
num_root                         int64
num_file_creations             float64
num_shells                       int64
num_access_files               float64
num_outbound_cmds                int64
is_host_login                    int64
is_guest_login                   i

## Identificação das classes

In [4]:
print("\nIDENTIFICAÇÃO DAS CLASSES")

# GF: Seleciona a coluna que contem os nomes das classes
classes = base["target"]

# GF: Conta o total de registros da base
total = len(classes)

# GF: Conta quantas vezes cada classe aparece
# GF: Organiza o resultado em uma tabela com as colunas "classe" e "quantidade"
tabela_classes = classes.value_counts().rename_axis("classe").reset_index(name="quantidade")

# GF: Calcula a porcentagem de registros de cada classe
# GN: round(4) é para até 4 casas decimais
tabela_classes["percentual"] = (100 * tabela_classes["quantidade"] / total).round(4)

# GF: Mostra o total de registros
print(f"Total de registros: {total}")

# GF: Mostra quantas classes diferentes existem na coluna target
print(f"Quantidade de classes originais: {classes.nunique()}")

# Exibe a tabela
display(tabela_classes)




IDENTIFICAÇÃO DAS CLASSES
Total de registros: 69833
Quantidade de classes originais: 2


,classe,quantidade,percentual
0,normal,65754,94.1589
1,anomaly,4079,5.8411


## Identificação de desbalanceamento

In [5]:
print("\nIDENTIFICAÇÃO DE DESBALANCEAMENTO")

# GF: Conta os registros de cada classe
contagem = base["target"].value_counts()

# GF: Mostra a classe mais frequente e a mais rara
print("Classe mais frequente:", contagem.index[0], "-", contagem.iloc[0], "registros")
print("Classe mais rara:", contagem.index[-1], "-", contagem.iloc[-1], "registros")

# GF: Calcula quantas vezes a maior classe supera a menor
print("Razão entre elas:", round(contagem.iloc[0] / contagem.iloc[-1], 1))


IDENTIFICAÇÃO DE DESBALANCEAMENTO
Classe mais frequente: normal - 65754 registros
Classe mais rara: anomaly - 4079 registros
Razão entre elas: 16.1


## Identificação de atributos categóricos

In [6]:

print("\n--- 1. IDENTIFICAÇÃO DE ATRIBUTOS CATEGÓRICOS ---")
# Seleciona as colunas baseadas no tipo de dado 'object' (strings) ou 'category'
colunas_categoricas = [col for col in base.select_dtypes(include=['object', 'category', 'string']).columns if col != 'target']

print("Atributos categóricos encontrados:")
for col in colunas_categoricas:
    valores_unicos = base[col].nunique()
    print(f"- {col} (Possui {valores_unicos} valores únicos)")

# Visualizando os valores unicos da coluna de alvo (target), por exemplo:
print("\nClasses do alvo:", base["target"].unique())


--- 1. IDENTIFICAÇÃO DE ATRIBUTOS CATEGÓRICOS ---
Atributos categóricos encontrados:
- protocol_type (Possui 6 valores únicos)
- service (Possui 96 valores únicos)
- flag (Possui 19 valores únicos)
- tipo_alerta_soc (Possui 2 valores únicos)

Classes do alvo: ['normal' 'anomaly']


In [7]:
import numpy as np

print("\nNUMÉRICOS QUE SE COMPORTAM COMO CATEGÓRICOS")

# Obtendo as colunas numericas dinamicamente a partir do DataFrame global 'base'
numericos = base.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Numericos com poucos valores distintos e todos inteiros sao candidatos a categoricos
limite = 5
for col in numericos:
    valores = base[col].dropna().unique()
    if 0 < len(valores) <= limite and np.all(np.mod(valores, 1) == 0):
        print(col, "->", len(valores), "valores:", [int(v) for v in sorted(valores)])


NUMÉRICOS QUE SE COMPORTAM COMO CATEGÓRICOS
land -> 2 valores: [0, 1]
wrong_fragment -> 3 valores: [0, 1, 3]
urgent -> 4 valores: [0, 1, 2, 3]
num_failed_logins -> 5 valores: [0, 1, 2, 3, 4]
logged_in -> 2 valores: [0, 1]
root_shell -> 2 valores: [0, 1]
su_attempted -> 3 valores: [0, 1, 2]
num_shells -> 3 valores: [0, 1, 2]
num_outbound_cmds -> 1 valores: [0]
is_host_login -> 2 valores: [0, 1]
is_guest_login -> 2 valores: [0, 1]
protocolo_cod -> 3 valores: [0, 1, 2]


## Identificação de incosnistências

In [8]:
print("\n--- 2. IDENTIFICAÇÃO DE INCONSISTÊNCIAS ---")

# Identificação de Valores Nulos/Ausentes
valores_nulos = base.isnull().sum()
colunas_com_nulos = valores_nulos[valores_nulos > 0]
if not colunas_com_nulos.empty:
    print("Valores ausentes encontrados nas seguintes colunas:")
    print(colunas_com_nulos)
else:
    print("Nenhum valor ausente (null/NaN) encontrado na base.")

# Verificação de domínios numéricos ilógicos
if 'duration' in base.columns:
    duracao_invalida = base[base['duration'] < 0]
    print(f"\nInstâncias com 'duration' negativa (inconsistente): {len(duracao_invalida)}")

# Taxas fora do intervalo esperado [0, 1]
colunas_taxa = [col for col in base.columns if col.endswith('_rate')]
taxas_invalidas = ((base[colunas_taxa] < 0) | (base[colunas_taxa] > 1)).sum()
print("\nTaxas fora de [0, 1]:")
print(taxas_invalidas[taxas_invalidas > 0])

# Verificação de anomalias estatísticas usando .describe()
# Isso ajuda a encontrar máximos ou mínimos impossíveis (ex: porcentagens > 1)
print("\nEstatísticas descritivas (para avaliar limites inconsistentes):")
print(base.describe().T[['min', 'max']])


--- 2. IDENTIFICAÇÃO DE INCONSISTÊNCIAS ---
Valores ausentes encontrados nas seguintes colunas:
service                         4416
flag                            4093
src_bytes                      27906
logged_in                       4250
num_compromised                 3395
root_shell                      3880
su_attempted                    3306
num_file_creations              3738
num_access_files                3172
count                           3724
srv_count                       3904
serror_rate                     4734
rerror_rate                     3743
srv_rerror_rate                 2923
same_srv_rate                   3571
diff_srv_rate                   3116
srv_diff_host_rate              4247
dst_host_count                  3963
dst_host_srv_count             27844
dst_host_diff_srv_rate          2410
dst_host_same_src_port_rate    19543
dst_host_serror_rate           21393
dst_host_srv_serror_rate        2786
dst_host_srv_rerror_rate        3298
src_bytes_kb   

## Identificação de Redundância

In [9]:
print("\n--- 3. IDENTIFICAÇÃO DE REDUNDÂNCIAS ---")

# Identificação de instâncias (linhas) duplicadas
qtd_duplicadas = base.duplicated().sum()
taxa_duplicadas = (qtd_duplicadas / len(base)) * 100
print(f"Linhas duplicadas encontradas: {qtd_duplicadas} ({taxa_duplicadas:.2f}% da base)")

# Caso queira remover as duplicadas:
# base_sem_redundancia = base.drop_duplicates()

# Identificação de redundância entre atributos (Correlação alta)
# Seleciona apenas as colunas numéricas
colunas_numericas = base.select_dtypes(include=['int64', 'float64'])

# Calcula a matriz de correlação de Pearson
matriz_corr = colunas_numericas.corr().abs()

# Filtra o triângulo superior da matriz para não mostrar pares espelhados (A-B e B-A)
import numpy as np
upper_tri = matriz_corr.where(np.triu(np.ones(matriz_corr.shape), k=1).astype(bool))

# Encontra features com correlação muito alta (ex: > 0.95), indicando redundância
features_redundantes = [col for col in upper_tri.columns if any(upper_tri[col] > 0.95)]
print(f"\nAtributos altamente correlacionados (redundantes - r > 0.95):")
print(features_redundantes)

formulas = {
    'src_bytes_kb': base['src_bytes'] / 1024,
    'dst_bytes_kb': base['dst_bytes'] / 1024,
    'total_bytes': base['src_bytes'] + base['dst_bytes'],
    'same_srv_pct': base['same_srv_rate'] * 100,
}
for coluna, esperado in formulas.items():
    validos = base[coluna].notna() & esperado.notna()
    iguais = np.isclose(base.loc[validos, coluna], esperado.loc[validos])
    print(f"{coluna}: {iguais.sum()} de {validos.sum()} valores disponiveis seguem a formula")


--- 3. IDENTIFICAÇÃO DE REDUNDÂNCIAS ---
Linhas duplicadas encontradas: 11355 (16.26% da base)

Atributos altamente correlacionados (redundantes - r > 0.95):
['srv_serror_rate', 'src_bytes_kb', 'dst_bytes_kb', 'same_srv_pct']
src_bytes_kb: 41927 de 41927 valores disponiveis seguem a formula
dst_bytes_kb: 69833 de 69833 valores disponiveis seguem a formula
total_bytes: 41927 de 41927 valores disponiveis seguem a formula
same_srv_pct: 66262 de 66262 valores disponiveis seguem a formula


## Identificação de Possíveis problemans de vazamento de dados

In [10]:
print("\n--- 4. IDENTIFICAÇÃO DE POSSÍVEIS PROBLEMAS DE VAZAMENTO DE DADOS ---")

import pandas as pd
import numpy as np

# Carrega o arquivo
df_vazamento = base.rename(columns={"target": "class"}).copy()

# Converte a classe em binário (0 = normal, 1 = anomaly) para investigar a correlação
alvo_binario = (df_vazamento['class'] != 'normal').astype(int)

# 1. Correlação de cada atributo numérico com a classe
colunas_num = [c for c in df_vazamento.select_dtypes(include=['int64', 'float64']).columns if df_vazamento[c].std() > 0]
correlacoes_alvo = df_vazamento[colunas_num].apply(lambda col: col.corr(alvo_binario)).abs().sort_values(ascending=False)

print("Top 10 atributos numéricos mais correlacionados com a classe ('class'):")
print(correlacoes_alvo.head(10).round(4))

# 2. Atributos suspeitos: verificar origem e disponibilidade antes de excluir
print("\nAtributos com suspeita de vazamento (investigar antes de remover):")

# A. tipo_alerta_soc
alerta_bin = (df_vazamento['tipo_alerta_soc'].astype(str).str.strip().str.upper() == 'ALERTA').astype(int)
print(f"\n1. 'tipo_alerta_soc' (Correlação com a classe: {alerta_bin.corr(alvo_binario):.4f}):")
print(pd.crosstab(df_vazamento['tipo_alerta_soc'], df_vazamento['class']))
print("   Associacao forte com a classe; verificar quando o alerta e produzido. Nao e o gabarito.")

# B. tempo_resposta_analista_min
presenca_tempo = df_vazamento['tempo_resposta_analista_min'].notnull().astype(int)
print(f"\n2. 'tempo_resposta_analista_min' (Correlação de preenchimento: {presenca_tempo.corr(alvo_binario):.4f} | Correlação de valor: {df_vazamento['tempo_resposta_analista_min'].corr(alvo_binario):.4f}):")
print("   Se for registrado apos o incidente, e informacao futura e deve ser excluido do modelo.")
print(f"   Presente em {presenca_tempo[df_vazamento['class'].eq('anomaly')].mean():.1%} das anomalias e ausente em {1 - presenca_tempo[df_vazamento['class'].eq('normal')].mean():.1%} dos normais.")

# C. score_ids_legado
print(f"\n3. 'score_ids_legado' (Correlação com a classe: {correlacoes_alvo['score_ids_legado']:.4f}):")
print(df_vazamento.groupby('class')['score_ids_legado'].agg(['count', 'mean', 'median', 'std']).round(4))
print("   Score associado a classe; verificar origem e se estava disponivel antes da previsao.")

# 3. Duplicatas no treino e risco para a validacao
qtd_duplicadas = df_vazamento.duplicated().sum()
print(f"4. Duplicatas encontradas no treino: {qtd_duplicadas} instâncias.")
print("Isso indica risco de vazamento se registros iguais caírem em partições diferentes da validação.")


--- 4. IDENTIFICAÇÃO DE POSSÍVEIS PROBLEMAS DE VAZAMENTO DE DADOS ---
Top 10 atributos numéricos mais correlacionados com a classe ('class'):
score_ids_legado               0.5734
dst_host_srv_serror_rate       0.5363
same_srv_rate                  0.5225
same_srv_pct                   0.5225
srv_serror_rate                0.5071
serror_rate                    0.4989
dst_host_serror_rate           0.4423
tempo_resposta_analista_min    0.3354
dst_host_srv_count             0.3017
logged_in                      0.2655
dtype: float64

Atributos com suspeita de vazamento (investigar antes de remover):

1. 'tipo_alerta_soc' (Correlação com a classe: 0.7282):
class            anomaly  normal
tipo_alerta_soc                 
ALERTA              3379    1657
OK                   700   64097
   Associacao forte com a classe; verificar quando o alerta e produzido. Nao e o gabarito.

2. 'tempo_resposta_analista_min' (Correlação de preenchimento: 0.6015 | Correlação de valor: 0.3354):
   Se for r

## Complemento — Categorias e diferenças de formatação
A comparação abaixo usa uma cópia das colunas textuais para investigar espaços e diferenças de caixa. Os valores de `base` permanecem preservados. As frequências incluem os valores ausentes.

A lista anterior de numéricos com poucos valores distintos indica candidatos à inspeção: contagens como `num_failed_logins`, `urgent` e `num_shells` continuam sendo quantitativas discretas. Poucos valores distintos, por si só, não tornam um atributo categórico.

In [11]:
categorias_padronizadas = pd.DataFrame(index=base.index)
resumo_categorias = []
for coluna in colunas_categoricas:
    original_texto = base[coluna].astype('string')
    padronizado = original_texto.str.strip()
    padronizado = padronizado.str.upper() if coluna in ['flag', 'tipo_alerta_soc'] else padronizado.str.lower()
    categorias_padronizadas[coluna] = padronizado
    resumo_categorias.append({
        'atributo': coluna,
        'categorias_originais': original_texto.nunique(),
        'categorias_padronizadas': padronizado.nunique(),
        'valores_ausentes': original_texto.isna().sum(),
        'registros_com_formato_diferente': original_texto.ne(padronizado).fillna(False).sum(),
    })
display(pd.DataFrame(resumo_categorias))

for coluna in colunas_categoricas:
    print(f'\nFrequências de {coluna}: valores originais e padronizados')
    # repr torna visíveis os espaços nas categorias originais.
    frequencias = pd.DataFrame({
        'original': base[coluna].astype('string').map(repr),
        'padronizado': categorias_padronizadas[coluna].fillna('<ausente>'),
    }).value_counts().rename('quantidade').reset_index()
    display(frequencias)

,atributo,categorias_originais,categorias_padronizadas,valores_ausentes,registros_com_formato_diferente
0,protocol_type,6,3,0,1455
1,service,96,67,4416,1531
2,flag,19,11,4093,1340
3,tipo_alerta_soc,2,2,0,0



Frequências de protocol_type: valores originais e padronizados


,original,padronizado,quantidade
0,'tcp',tcp,54824
1,'udp',udp,11940
2,'icmp',icmp,1614
3,'TCP',tcp,1186
4,'UDP',udp,240
5,'ICMP',icmp,29



Frequências de service: valores originais e padronizados


,original,padronizado,quantidade
0,'http',http,35551
1,'domain_u',domain_u,7497
2,'smtp',smtp,5945
3,<NA>,<ausente>,4416
4,'ftp_data',ftp_data,4190
...,...,...,...
92,'exec ',exec,1
93,'ntp_u ',ntp_u,1
94,'http_2784',http_2784,1
95,'supdup ',supdup,1



Frequências de flag: valores originais e padronizados


,original,padronizado,quantidade
0,'SF',SF,58058
1,<NA>,<ausente>,4093
2,'REJ',REJ,2856
3,'S0',S0,2399
4,'sf',SF,1220
5,'RSTR',RSTR,328
6,'S1',S1,288
7,'RSTO',RSTO,287
8,'S2',S2,103
9,'rej',REJ,61



Frequências de tipo_alerta_soc: valores originais e padronizados


,original,padronizado,quantidade
0,'OK',OK,64797
1,'ALERTA',ALERTA,5036


## Complemento — Valores negativos de bytes
Contagens de bytes negativos são inconsistentes com a interpretação de quantidade de dados transmitidos. Esta verificação apenas quantifica os casos; a escolha do tratamento será feita na Etapa 02.

In [12]:
bytes_negativos = base['src_bytes'].lt(0)
print(f"Registros com src_bytes negativo: {bytes_negativos.sum()}")
print(f"Percentual do treino: {100 * bytes_negativos.mean():.4f}%")
print(f"Menor src_bytes observado: {base['src_bytes'].min()}")

Registros com src_bytes negativo: 189
Percentual do treino: 0.2706%
Menor src_bytes observado: -2044.0


## Complemento — Atributos constantes e codificação redundante
A tabela identifica colunas com um único valor observado, contabilizando ausências separadamente. O cruzamento entre protocolo padronizado e código verifica se `protocolo_cod` repete a informação de `protocol_type`.

In [13]:
quantidades_distintas = base.drop(columns='target').nunique(dropna=True)
atributos_constantes = quantidades_distintas[quantidades_distintas.eq(1)].index
resumo_constantes = pd.DataFrame({
    'atributo': atributos_constantes,
    'valor_observado': [base[coluna].dropna().iloc[0] for coluna in atributos_constantes],
    'ausentes': [base[coluna].isna().sum() for coluna in atributos_constantes],
})
display(resumo_constantes)

print('Protocolo padronizado versus protocolo_cod:')
tabela_protocolo_codigo = pd.crosstab(
    categorias_padronizadas['protocol_type'], base['protocolo_cod'], dropna=False
)
display(tabela_protocolo_codigo)
pares_protocolo_codigo = pd.DataFrame({
    'protocolo': categorias_padronizadas['protocol_type'],
    'codigo': base['protocolo_cod'],
}).dropna().drop_duplicates()
correspondencia_biunivoca = (
    not pares_protocolo_codigo.empty
    and pares_protocolo_codigo.groupby('protocolo')['codigo'].nunique().eq(1).all()
    and pares_protocolo_codigo.groupby('codigo')['protocolo'].nunique().eq(1).all()
)
print(f'Correspondência de um para um nos pares observados: {correspondencia_biunivoca}')

,atributo,valor_observado,ausentes
0,num_outbound_cmds,0,0


Protocolo padronizado versus protocolo_cod:


protocolo_cod,0,1,2
protocol_type,,,
icmp,0,0,1643
tcp,56010,0,0
udp,0,12180,0


Correspondência de um para um nos pares observados: True


## Complemento — Atributos iguais com classes conflitantes
A análise agrupa linhas pelos 49 atributos descritivos originais, considerando também os padrões de ausência. Um grupo com as duas classes indica conflito de rótulos nos atributos observados. Isso não permite determinar automaticamente qual rótulo está correto ou a origem do conflito. Nenhuma linha é removida.

In [14]:
atributos_para_conflito = base.columns.drop('target').tolist()
grupos_rotulos = base.groupby(
    atributos_para_conflito, dropna=False, sort=False
)['target'].agg(quantidade='size', classes_distintas='nunique')
grupos_conflitantes = grupos_rotulos[grupos_rotulos['classes_distintas'].gt(1)]
print(f'Grupos de atributos iguais com classes conflitantes: {len(grupos_conflitantes)}')
print(f"Linhas nesses grupos: {grupos_conflitantes['quantidade'].sum()}")
print(f"Percentual do treino: {100 * grupos_conflitantes['quantidade'].sum() / len(base):.4f}%")
print(f'Duplicatas exatas além da primeira ocorrência, incluindo a classe: {base.duplicated().sum()}')

Grupos de atributos iguais com classes conflitantes: 1702
Linhas nesses grupos: 3730
Percentual do treino: 5.3413%
Duplicatas exatas além da primeira ocorrência, incluindo a classe: 11355


## Complemento — Associação de todos os atributos com a classe
A tabela numérica mantém o sinal da correlação de Pearson com `anomaly = 1` e `normal = 0`, além de seu valor absoluto e da quantidade de observações válidas. A correlação de uma coluna constante é indefinida (`NaN`). Para os atributos textuais, são exibidas contagens por classe e taxas de anomalia por categoria padronizada, incluindo ausências.

Associação estatística forte é um sinal para investigação. A confirmação de vazamento depende da origem da informação e de sua disponibilidade no momento da previsão. Essas verificações usam somente o treino.

In [15]:
alvo_complemento = base['target'].eq('anomaly').astype(int)
atributos_numericos_complemento = base.drop(columns='target').select_dtypes(include='number').columns
linhas_correlacao = []
for coluna in atributos_numericos_complemento:
    valores = base[coluna]
    correlacao = valores.corr(alvo_complemento) if valores.nunique(dropna=True) > 1 else np.nan
    linhas_correlacao.append({
        'atributo': coluna,
        'correlacao_pearson': correlacao,
        'correlacao_absoluta': abs(correlacao),
        'observacoes_validas': valores.notna().sum(),
    })
tabela_correlacoes_completa = pd.DataFrame(linhas_correlacao).sort_values(
    'correlacao_absoluta', ascending=False, na_position='last'
)
print('Todos os atributos numéricos, incluindo os constantes:')
display(tabela_correlacoes_completa.round(4))

for coluna in colunas_categoricas:
    categorias = categorias_padronizadas[coluna].fillna('<ausente>')
    associacao = pd.crosstab(categorias, base['target']).reindex(
        columns=['normal', 'anomaly'], fill_value=0
    )
    associacao['total'] = associacao[['normal', 'anomaly']].sum(axis=1)
    associacao['percentual_anomalias'] = 100 * associacao['anomaly'] / associacao['total']
    print(f'\nDistribuição das classes por categoria de {coluna}:')
    display(associacao.sort_values('total', ascending=False).round(4))

Todos os atributos numéricos, incluindo os constantes:


,atributo,correlacao_pearson,correlacao_absoluta,observacoes_validas
43,score_ids_legado,0.5734,0.5734,69833
35,dst_host_srv_serror_rate,0.5363,0.5363,67047
25,same_srv_rate,-0.5225,0.5225,66262
41,same_srv_pct,-0.5225,0.5225,66262
22,srv_serror_rate,0.5071,0.5071,69833
21,serror_rate,0.4989,0.4989,65099
34,dst_host_serror_rate,0.4423,0.4423,48440
44,tempo_resposta_analista_min,0.3354,0.3354,6029
29,dst_host_srv_count,-0.3017,0.3017,41989
8,logged_in,-0.2655,0.2655,65583



Distribuição das classes por categoria de protocol_type:


target,normal,anomaly,total,percentual_anomalias
protocol_type,,,,
tcp,52561,3449,56010,6.1578
udp,11933,247,12180,2.0279
icmp,1260,383,1643,23.3110



Distribuição das classes por categoria de service:


target,normal,anomaly,total,percentual_anomalias
service,,,,
http,35612,606,36218,1.6732
domain_u,7597,74,7671,0.9647
smtp,5992,78,6070,1.2850
<ausente>,4141,275,4416,6.2274
ftp_data,4141,128,4269,2.9984
...,...,...,...,...
printer,2,2,4,50.0000
tftp_u,4,0,4,0.0000
red_i,4,0,4,0.0000



Distribuição das classes por categoria de flag:


target,normal,anomaly,total,percentual_anomalias
flag,,,,
SF,57965,1313,59278,2.2150
<ausente>,3848,245,4093,5.9858
REJ,2309,608,2917,20.8433
S0,784,1648,2432,67.7632
RSTR,193,145,338,42.8994
S1,291,1,292,0.3425
RSTO,209,82,291,28.1787
S2,104,5,109,4.5872
S3,41,13,54,24.0741



Distribuição das classes por categoria de tipo_alerta_soc:


target,normal,anomaly,total,percentual_anomalias
tipo_alerta_soc,,,,
OK,64097,700,64797,1.0803
ALERTA,1657,3379,5036,67.0969
